In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"

FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_DIR / "london_underground_clean.csv")

In [2]:
# 1. Basic data audit

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nRows per year:")
print(df.groupby("Year").size())

print("\nUnique stations per year:")
print(df.groupby("Year")["Station"].nunique())

print("\nUnique station codes per year:")
print(df.groupby("Year")["MNLC"].nunique())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate station codes per year:")
print(
    df.groupby("Year")["MNLC"]
      .apply(lambda x: x.duplicated().sum())
)

Dataset shape: (807, 16)

Columns:
['Year', 'Mode', 'MNLC', 'MASC', 'Station', 'Coverage', 'Source', 'MTT_Entry', 'Friday_Entry', 'Saturday_Entry', 'Sunday_Entry', 'MTT_Exit', 'Friday_Exit', 'Saturday_Exit', 'Sunday_Exit', 'Annualised_EnEx']

Rows per year:
Year
2019    269
2022    269
2025    269
dtype: int64

Unique stations per year:
Year
2019    269
2022    269
2025    269
Name: Station, dtype: int64

Unique station codes per year:
Year
2019    269
2022    269
2025    269
Name: MNLC, dtype: int64

Missing values:
Year               0
Mode               0
MNLC               0
MASC               0
Station            0
Coverage           0
Source             0
MTT_Entry          2
Friday_Entry       2
Saturday_Entry     2
Sunday_Entry       2
MTT_Exit           2
Friday_Exit        2
Saturday_Exit      2
Sunday_Exit        2
Annualised_EnEx    2
dtype: int64

Duplicate station codes per year:
Year
2019    0
2022    0
2025    0
Name: MNLC, dtype: int64


In [3]:
# 2. Data preparation

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

passenger_columns = [
    "MTT_Entry", "Friday_Entry", "Saturday_Entry", "Sunday_Entry",
    "MTT_Exit", "Friday_Exit", "Saturday_Exit", "Sunday_Exit",
    "Annualised_EnEx"
]

for col in passenger_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Standardise station codes
df["MNLC"] = (
    df["MNLC"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Create a matching key:
# Use MNLC where available, otherwise use the station name
df["Station_ID"] = df["MNLC"]

missing_code = df["Station_ID"].isna() | (df["Station_ID"] == "")

df.loc[missing_code, "Station_ID"] = (
    df.loc[missing_code, "Station"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Data types:")
print(df[["Year", "MNLC", "Station_ID"] + passenger_columns[:4]].dtypes)

print("\nMissing station IDs:", df["Station_ID"].isna().sum())

Data types:
Year                Int64
MNLC               string
Station_ID         string
MTT_Entry         float64
Friday_Entry      float64
Saturday_Entry    float64
Sunday_Entry      float64
dtype: object

Missing station IDs: 0


In [4]:
# 3. Identify stations available in all three years

years = [2019, 2022, 2025]

station_year_counts = (
    df[df["Year"].isin(years)]
    .groupby("Station_ID")["Year"]
    .nunique()
)

common_station_ids = station_year_counts[
    station_year_counts == len(years)
].index

# Require valid baseline and comparison passenger data
required_columns = [
    "MTT_Entry", "MTT_Exit",
    "Friday_Entry", "Friday_Exit",
    "Saturday_Entry", "Saturday_Exit",
    "Sunday_Entry", "Sunday_Exit"
]

comparable_df = df[
    df["Station_ID"].isin(common_station_ids)
].copy()

comparable_df = comparable_df.dropna(subset=required_columns)

print("Comparable station records:")
print(comparable_df.groupby("Year").size())

print("\nComparable unique stations:")
print(comparable_df.groupby("Year")["Station_ID"].nunique())

Comparable station records:
Year
2019    267
2022    269
2025    269
dtype: int64

Comparable unique stations:
Year
2019    267
2022    269
2025    269
Name: Station_ID, dtype: int64


In [5]:
# 4. Calculate passenger flow by day type

day_types = {
    "Weekday (Mon-Thu)": ("MTT_Entry", "MTT_Exit"),
    "Friday": ("Friday_Entry", "Friday_Exit"),
    "Saturday": ("Saturday_Entry", "Saturday_Exit"),
    "Sunday": ("Sunday_Entry", "Sunday_Exit")
}

for day, (entry_col, exit_col) in day_types.items():
    comparable_df[f"{day}_Flow"] = (
        comparable_df[entry_col] + comparable_df[exit_col]
    )

flow_columns = [f"{day}_Flow" for day in day_types]

print(comparable_df[
    ["Year", "Station", "Station_ID"] + flow_columns
].head())

   Year       Station Station_ID  Weekday (Mon-Thu)_Flow  Friday_Flow  \
0  2019    Acton Town        500                 20114.0      20381.0   
1  2019       Aldgate        502                 35391.0      32184.0   
2  2019  Aldgate East        503                 45444.0      48548.0   
3  2019      Alperton        505                  9384.0       9391.0   
4  2019      Amersham        506                  8184.0       7597.0   

   Saturday_Flow  Sunday_Flow  
0        15162.0      10679.0  
1        16838.0       9652.0  
2        34537.0      27345.0  
3         6529.0       4958.0  
4         4286.0       2814.0  


In [6]:
# 5. Aggregate passenger flow by year and day type

overall_flow = (
    comparable_df
    .groupby("Year")[flow_columns]
    .sum()
)

print(overall_flow)

      Weekday (Mon-Thu)_Flow   Friday_Flow  Saturday_Flow   Sunday_Flow
Year                                                                   
2019            9.357410e+06  9.609363e+06   6.975112e+06  4.748107e+06
2022            7.146592e+06  6.934478e+06   6.292816e+06  4.256133e+06
2025            7.760794e+06  7.261145e+06   6.703083e+06  4.547586e+06


In [7]:
# 6. Calculate recovery index

recovery_index = overall_flow.div(
    overall_flow.loc[2019]
).mul(100)

print("\nPassenger Recovery Index (% of 2019):")
print(recovery_index.round(1))


Passenger Recovery Index (% of 2019):
      Weekday (Mon-Thu)_Flow  Friday_Flow  Saturday_Flow  Sunday_Flow
Year                                                                 
2019                   100.0        100.0          100.0        100.0
2022                    76.4         72.2           90.2         89.6
2025                    82.9         75.6           96.1         95.8


In [8]:
# %pip install nbformat

In [9]:
# 7. Plot overall recovery trends

recovery_plot = (
    recovery_index
    .reset_index()
    .melt(
        id_vars="Year",
        var_name="Day Type",
        value_name="Recovery Index"
    )
)

fig = px.line(
    recovery_plot,
    x="Year",
    y="Recovery Index",
    color="Day Type",
    markers=True,
    title="London Underground Passenger Recovery, 2019–2025",
    labels={
        "Year": "Year",
        "Recovery Index": "Passenger Recovery (% of 2019)"
    }
)

fig.add_hline(
    y=100,
    line_dash="dash",
    line_color="gray",
    annotation_text="2019 baseline"
)

fig.update_layout(
    yaxis_title="Passenger Recovery (% of 2019)",
    xaxis=dict(tickmode="array", tickvals=[2019, 2022, 2025]),
    template="plotly_white"
)

fig.show()

fig.write_html(FIGURE_DIR / "overall_recovery.html")

In [10]:
# 8. Station-level weekday passenger flow comparison

weekday_comparison = comparable_df.pivot(
    index=["Station_ID", "Station"],
    columns="Year",
    values="Weekday (Mon-Thu)_Flow"
).reset_index()

weekday_comparison = weekday_comparison.dropna(
    subset=[2019, 2025]
)

weekday_comparison["Change (%)"] = (
    (weekday_comparison[2025] / weekday_comparison[2019]) - 1
) * 100

weekday_comparison["Recovery Index"] = (
    weekday_comparison[2025] / weekday_comparison[2019]
) * 100

weekday_comparison = weekday_comparison.sort_values(
    "Change (%)",
    ascending=False
)

print("Top 10 stations with the highest growth:")
print(weekday_comparison.head(10).round(1))

print("\nTop 10 stations with the largest decline:")
print(weekday_comparison.tail(10).sort_values("Change (%)").round(1))

weekday_comparison.to_csv(
    TABLE_DIR / "station_weekday_recovery.csv",
    index=False
)

Top 10 stations with the highest growth:
Year Station_ID               Station      2019      2022      2025  \
71          577            Farringdon   98706.0  119436.0  163498.6   
173         686              Richmond   24618.0   11662.9   39607.0   
214         728  Tottenham Court Road  132723.0  158194.2  191246.4   
249         767             Wimbledon   40525.0   38176.9   56818.9   
245         763           Whitechapel   44865.0   49736.3   61081.2   
239         757              West Ham   14609.0   15353.0   17986.3   
157         670        Paddington TfL  158116.0  148054.3  191374.5   
22          524           Bond Street  120419.0  119912.1  144075.1   
136         645              Moorgate   94115.0   80702.9  112149.3   
29          531        Buckhurst Hill    6275.0    5819.3    7431.1   

Year  Change (%)  Recovery Index  
71          65.6           165.6  
173         60.9           160.9  
214         44.1           144.1  
249         40.2           140.2  
24

In [11]:
# # Find station records with missing passenger data
# missing_records = df[
#     df[required_columns].isna().any(axis=1)
# ][
#     ["Year", "Station", "MNLC", "Source"] + required_columns
# ]

# print("Records with missing passenger data:")
# print(missing_records.to_string(index=False))

# print("\nMissing records by year:")
# print(missing_records.groupby("Year").size())

In [12]:
# Keep only stations with complete passenger data in all three years

complete_records = df.dropna(subset=required_columns).copy()

complete_station_counts = (
    complete_records.groupby("Station_ID")["Year"].nunique()
)

complete_station_ids = complete_station_counts[
    complete_station_counts == 3
].index

balanced_df = complete_records[
    complete_records["Station_ID"].isin(complete_station_ids)
].copy()

print("Balanced station panel:")
print(balanced_df.groupby("Year").size())

print("\nUnique stations per year:")
print(balanced_df.groupby("Year")["Station_ID"].nunique())

Balanced station panel:
Year
2019    267
2022    267
2025    267
dtype: int64

Unique stations per year:
Year
2019    267
2022    267
2025    267
Name: Station_ID, dtype: int64


In [13]:
day_types = {
    "Weekday (Mon-Thu)": ("MTT_Entry", "MTT_Exit"),
    "Friday": ("Friday_Entry", "Friday_Exit"),
    "Saturday": ("Saturday_Entry", "Saturday_Exit"),
    "Sunday": ("Sunday_Entry", "Sunday_Exit")
}

for day, (entry_col, exit_col) in day_types.items():
    balanced_df[f"{day}_Flow"] = (
        balanced_df[entry_col] + balanced_df[exit_col]
    )

flow_columns = [f"{day}_Flow" for day in day_types]

overall_flow = balanced_df.groupby("Year")[flow_columns].sum()

recovery_index = (
    overall_flow.div(overall_flow.loc[2019]).mul(100)
)

print("Overall passenger flow:")
print(overall_flow.round(0))

print("\nPassenger Recovery Index (% of 2019):")
print(recovery_index.round(1))

Overall passenger flow:
      Weekday (Mon-Thu)_Flow  Friday_Flow  Saturday_Flow  Sunday_Flow
Year                                                                 
2019               9357410.0    9609363.0      6975112.0    4748107.0
2022               7112256.0    6896307.0      6241593.0    4225218.0
2025               7720954.0    7223485.0      6660482.0    4517007.0

Passenger Recovery Index (% of 2019):
      Weekday (Mon-Thu)_Flow  Friday_Flow  Saturday_Flow  Sunday_Flow
Year                                                                 
2019                   100.0        100.0          100.0        100.0
2022                    76.0         71.8           89.5         89.0
2025                    82.5         75.2           95.5         95.1


In [14]:
recovery_plot = (
    recovery_index.reset_index()
    .melt(
        id_vars="Year",
        var_name="Day Type",
        value_name="Recovery Index"
    )
)

fig = px.line(
    recovery_plot,
    x="Year",
    y="Recovery Index",
    color="Day Type",
    markers=True,
    title="London Underground Passenger Recovery, 2019–2025",
    labels={
        "Year": "Year",
        "Recovery Index": "Passenger Recovery (% of 2019)"
    }
)

fig.add_hline(
    y=100,
    line_dash="dash",
    line_color="gray",
    annotation_text="2019 baseline"
)

fig.update_layout(
    yaxis_title="Passenger Recovery (% of 2019)",
    xaxis=dict(tickmode="array", tickvals=[2019, 2022, 2025]),
    template="plotly_white"
)

fig.show()
fig.write_html(FIGURE_DIR / "overall_recovery.html")

In [15]:
# Compare weekday passenger flow using station codes only
weekday_comparison = balanced_df.pivot(
    index="Station_ID",
    columns="Year",
    values="Weekday (Mon-Thu)_Flow"
).reset_index()

# Get one consistent station name per station code
station_names = (
    balanced_df[balanced_df["Year"] == 2025]
    [["Station_ID", "Station"]]
    .drop_duplicates(subset="Station_ID")
)

weekday_comparison = weekday_comparison.merge(
    station_names,
    on="Station_ID",
    how="left"
)

# Calculate percentage change and recovery index
weekday_comparison["Change (%)"] = (
    weekday_comparison[2025] / weekday_comparison[2019] - 1
) * 100

weekday_comparison["Recovery Index"] = (
    weekday_comparison[2025] / weekday_comparison[2019]
) * 100

weekday_comparison = weekday_comparison.sort_values(
    "Change (%)",
    ascending=False
)

print("Number of stations:", len(weekday_comparison))
print("Missing values:")
print(weekday_comparison[[2019, 2022, 2025]].isna().sum())

print("\nTop 10 stations with the highest growth:")
print(
    weekday_comparison.head(10)
    [["Station", 2019, 2022, 2025, "Change (%)", "Recovery Index"]]
    .round(1)
    .to_string(index=False)
)

print("\nTop 10 stations with the largest decline:")
print(
    weekday_comparison.tail(10)
    .sort_values("Change (%)")
    [["Station", 2019, 2022, 2025, "Change (%)", "Recovery Index"]]
    .round(1)
    .to_string(index=False)
)

weekday_comparison.to_csv(
    TABLE_DIR / "station_weekday_recovery.csv",
    index=False
)

Number of stations: 267
Missing values:
2019    0
2022    0
2025    0
dtype: int64

Top 10 stations with the highest growth:
             Station     2019     2022     2025  Change (%)  Recovery Index
          Farringdon  98706.0 119436.0 163498.6        65.6           165.6
            Richmond  24618.0  11662.9  39607.0        60.9           160.9
Tottenham Court Road 132723.0 158194.2 191246.4        44.1           144.1
           Wimbledon  40525.0  38176.9  56818.9        40.2           140.2
         Whitechapel  44865.0  49736.3  61081.2        36.1           136.1
            West Ham  14609.0  15353.0  17986.3        23.1           123.1
      Paddington TfL 158116.0 148054.3 191374.5        21.0           121.0
         Bond Street 120419.0 119912.1 144075.1        19.6           119.6
            Moorgate  94115.0  80702.9 112149.3        19.2           119.2
      Buckhurst Hill   6275.0   5819.3   7431.1        18.4           118.4

Top 10 stations with the largest decli

In [16]:
extension_stations = [
    "Nine Elms",
    "Battersea Power Station"
]

extension_df = df[
    df["Station"].isin(extension_stations)
].copy()

extension_df["Total Weekday Flow"] = (
    extension_df["MTT_Entry"] + extension_df["MTT_Exit"]
)

extension_summary = extension_df.pivot(
    index="Station",
    columns="Year",
    values="Total Weekday Flow"
).reset_index()

extension_summary["Change 2022-2025 (%)"] = (
    (extension_summary[2025] / extension_summary[2022]) - 1
) * 100

print(extension_summary.round(1).to_string(index=False))

extension_summary.to_csv(
    TABLE_DIR / "northern_line_extension.csv",
    index=False
)

                Station  2019    2022    2025  Change 2022-2025 (%)
Battersea Power Station   NaN 23055.1 26589.6                  15.3
              Nine Elms   NaN 11281.0 13250.4                  17.5


In [17]:
# Step 12 — Classify stations by recovery level

weekday_comparison["Recovery Group"] = pd.cut(
    weekday_comparison["Recovery Index"],
    bins=[-float("inf"), 90, 100, float("inf")],
    labels=[
        "Below baseline",
        "Near baseline",
        "Above baseline"
    ],
    right=False
)

# Count stations in each group
recovery_groups = (
    weekday_comparison["Recovery Group"]
    .value_counts()
    .reindex([
        "Above baseline",
        "Near baseline",
        "Below baseline"
    ])
    .rename("Station Count")
    .to_frame()
)

recovery_groups["Percentage (%)"] = (
    recovery_groups["Station Count"]
    / len(weekday_comparison) * 100
)

print("Station Recovery Distribution:")
print(recovery_groups.round(1))

# Save classification table
weekday_comparison.to_csv(
    TABLE_DIR / "station_recovery_classification.csv",
    index=False
)

recovery_groups.to_csv(
    TABLE_DIR / "station_recovery_groups.csv"
)


# Step 13 — Visualise recovery distribution

fig = px.bar(
    recovery_groups.reset_index(),
    x="Recovery Group",
    y="Percentage (%)",
    text="Station Count",
    title="How Evenly Has Passenger Recovery Been Distributed?",
    labels={
        "Recovery Group": "Station Recovery Category",
        "Percentage (%)": "Share of Stations"
    },
    category_orders={
        "Recovery Group": [
            "Above baseline",
            "Near baseline",
            "Below baseline"
        ]
    }
)

fig.update_traces(
    textposition="outside",
    texttemplate="%{text} stations (%{y:.1f}%)"
)

fig.update_layout(
    template="plotly_white",
    yaxis_title="Share of Stations (%)",
    xaxis_title="",
    showlegend=False
)

fig.show()

fig.write_html(
    FIGURE_DIR / "station_recovery_groups.html"
)

print("\nFiles saved successfully!")

Station Recovery Distribution:
                Station Count  Percentage (%)
Recovery Group                               
Above baseline             21             7.9
Near baseline              23             8.6
Below baseline            223            83.5



Files saved successfully!
